# Matched-protocol evaluation: TS-SatFire reference metric vs. micro-averaged metric

Inference only. No training. Designed for Kaggle with 4x NVIDIA T4.
Typical runtime: under 45 minutes total, far inside the 12-hour session limit.

Purpose: score the existing SpaSE-UNet3D checkpoints (AF and FP) under
  (a) the TS-SatFire reference evaluation code, reimplemented verbatim, and
  (b) our own pixel-level micro-averaged metric,
on both the full official test sets and the audited subsets, so that every
number in the paper can be attributed to a stated protocol.

Reference protocol reproduced from zhaoyutim/TS-SatFire:
  run_spatial_temp_model.py lines 299-357, run_spatial_temp_model_pred.py lines 295-362,
  satimg_dataset_processor/satimg_dataset_processor.py lines 55-115,
  satimg_dataset_processor/data_generator_torch.py lines 36-95.
Key properties of that protocol:
  1. sklearn f1_score / jaccard_score computed per 256x256 frame, zero_division=1.0
  2. predictions forced to 0 where label == -1 (BA/FP nodata sentinel), label then > 0
  3. AF label is nan_to_num(band7) > 0 (NaN becomes 0, any nonzero confidence is positive)
  4. fixed decision threshold 0.5, no sweep
  5. macro-average over frames within a fire, then unweighted macro-average over fires
  6. AF/BA test windows use stride = TS; prediction test windows use stride = 1
  7. all TS frames of a window are scored for AF/BA, not only the final day


In [1]:
import os, sys, glob, json, math, time, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score, jaccard_score

warnings.filterwarnings("ignore")

try:
    import rasterio
    HAS_RASTERIO = True
except Exception:
    import tifffile
    HAS_RASTERIO = False

N_GPU = torch.cuda.device_count()
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPUs:", N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print("  GPU {}: {} -- {:.1f} GB".format(i, p.name, p.total_memory / 1e9))

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


PyTorch: 2.10.0+cu128 | CUDA: 12.8 | GPUs: 2
  GPU 0: Tesla T4 -- 15.6 GB
  GPU 1: Tesla T4 -- 15.6 GB


## 1. Configuration


In [2]:
class CFG:
    CROP = 256
    AF_TS = 2
    FP_TS = 2

    AF_CHANNELS = 8
    FP_CHANNELS = 27
    FP_BAND_IDX = [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

    SAT_MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                         294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    SAT_STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                        24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    # Reference protocol threshold. Our own protocol thresholds are applied separately.
    REF_THRESHOLD = 0.50
    OUR_AF_THRESHOLD = 0.20
    OUR_FP_THRESHOLD = 0.55

    # Full official AF test set, in the order used by the reference repository.
    AF_TEST_ALL = [
        "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire", "sparks_lake_fire",
        "lytton_fire", "chuckegg_creek_fire", "swedish_fire", "sydney_fire",
        "thomas_fire", "tubbs_fire", "carr_fire", "camp_fire",
        "creek_fire", "blue_ridge_fire", "dixie_fire", "mosquito_fire", "calfcanyon_fire",
    ]
    AF_TEST_EXCLUDE = ["mosquito_fire", "calfcanyon_fire"]

    # Fires removed from the FP test aggregate in the submitted manuscript.
    FP_TEST_EXCLUDE = [
        "US_2021_FL2521008104520210308",
        "US_2021_MT4714310953420211004",
        "US_2021_NM3323810847220210520",
        "US_2021_NM3340210587120210426",
        "US_2021_NM3344410803520210514",
        "US_2021_NM3676810505920211120",
        "US_2021_AZ3345510938920210616",
        "US_2021_AZ3368910927620210616",
    ]


def find_data_root():
    cands = [
        "/kaggle/input/ts-satfire/ts-satfire/ts-satfire/",
        "/kaggle/input/ts-satfire/ts-satfire/",
        "/kaggle/input/ts-satfire/",
    ]
    for p in cands:
        if os.path.isdir(p):
            items = os.listdir(p)
            if any(d[:1].isdigit() or d.startswith("US_2021") or d.endswith("_fire") for d in items):
                return p
    for root, dirs, _ in os.walk("/kaggle/input"):
        if any(d.endswith("_fire") for d in dirs) and any(d.isdigit() for d in dirs):
            return root
        if root.replace("/kaggle/input", "").count(os.sep) >= 5:
            dirs.clear()
    return None


def find_checkpoints():
    """Return every checkpoint-like file outside the dataset tree, largest first."""
    exts = (".pt", ".pth", ".ckpt")
    found, seen = [], set()
    ds = os.path.abspath(DATA_ROOT) if DATA_ROOT else None
    for base in ["/kaggle/input", "/kaggle/working", "/kaggle/temp"]:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if ds and os.path.abspath(root).startswith(ds):
                dirs.clear(); continue
            for f in files:
                if f.endswith(exts):
                    full = os.path.join(root, f)
                    if full in seen:
                        continue
                    seen.add(full)
                    try:
                        found.append((full, os.path.getsize(full) / 1e6))
                    except OSError:
                        pass
    found.sort(key=lambda x: -x[1])
    return found


def pick_ckpt(cands, keys):
    for p, _ in cands:
        low = os.path.basename(p).lower()
        if any(k in low for k in keys):
            return p
    return None


DATA_ROOT = find_data_root()
assert DATA_ROOT, "TS-SatFire dataset not found under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

CKPTS = find_checkpoints()
print("\nCheckpoints found:")
for p, s in CKPTS:
    print("  {:.1f} MB  {}".format(s, p))

assert CKPTS, (
    "No checkpoints found. Attach the spase-unet3d-checkpoints dataset "
    "(Add Input -> Datasets), then rerun."
)

AF_CKPT = os.environ.get("AF_CKPT") or pick_ckpt(CKPTS, ["af", "active"]) \
    or (CKPTS[0][0] if CKPTS else None)
FP_CKPT = os.environ.get("FP_CKPT") or pick_ckpt(CKPTS, ["fp", "pred", "prog"]) \
    or (CKPTS[1][0] if len(CKPTS) > 1 else None)

print("\nAF_CKPT:", AF_CKPT)
print("FP_CKPT:", FP_CKPT)
assert AF_CKPT and os.path.isfile(AF_CKPT), "AF checkpoint not resolved"
assert FP_CKPT and os.path.isfile(FP_CKPT), "FP checkpoint not resolved"
assert AF_CKPT != FP_CKPT, "AF and FP resolved to the same file; set them manually"


DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire

Checkpoints found:
  131.6 MB  /kaggle/input/datasets/nmavros/spase-unet3d-checkpoints/af_ts2_best.pt
  97.2 MB  /kaggle/input/datasets/nmavros/spase-unet3d-checkpoints/fp_v6_best.pth

AF_CKPT: /kaggle/input/datasets/nmavros/spase-unet3d-checkpoints/af_ts2_best.pt
FP_CKPT: /kaggle/input/datasets/nmavros/spase-unet3d-checkpoints/fp_v6_best.pth


## 2. The two metrics

`ref_frame_score` is a line-for-line reimplementation of the reference repository's
per-frame scoring. `MicroAccumulator` is our pixel-level aggregation.


In [3]:
def ref_frame_score(label, pred):
    """One 256x256 frame under the TS-SatFire reference protocol.

    label : float array, may contain the -1 nodata sentinel
    pred  : boolean/0-1 array already thresholded at REF_THRESHOLD

    Returns (f1, iou, is_empty_gt, is_free_score).
    is_free_score marks frames that score 1.0 purely because both the label and
    the prediction are empty, which is the zero_division=1.0 branch of sklearn.
    """
    pred = np.where(label == -1, 0, pred)
    y_true = (label > 0).astype(np.uint8).ravel()
    y_pred = np.asarray(pred).astype(np.uint8).ravel()

    f1 = f1_score(y_true, y_pred, zero_division=1.0)
    iou = jaccard_score(y_true, y_pred, zero_division=1.0)

    empty_gt = bool(y_true.sum() == 0)
    free = bool(empty_gt and y_pred.sum() == 0)
    return float(f1), float(iou), empty_gt, free


class MicroAccumulator:
    """Pixel-level TP/FP/FN accumulation; F1 and IoU satisfy IoU = F1 / (2 - F1)."""

    def __init__(self):
        self.tp = self.fp = self.fn = 0

    def add(self, label_bin, pred_bin, valid=None):
        lbl = np.asarray(label_bin).astype(bool)
        prd = np.asarray(pred_bin).astype(bool)
        if valid is not None:
            m = np.asarray(valid).astype(bool)
            lbl, prd = lbl & m, prd & m
        self.tp += int((prd & lbl).sum())
        self.fp += int((prd & ~lbl).sum())
        self.fn += int((~prd & lbl).sum())

    @property
    def f1(self):
        d = 2 * self.tp + self.fp + self.fn
        return (2.0 * self.tp / d) if d else float("nan")

    @property
    def iou(self):
        d = self.tp + self.fp + self.fn
        return (float(self.tp) / d) if d else float("nan")

    def as_dict(self):
        return {"tp": self.tp, "fp": self.fp, "fn": self.fn,
                "f1": self.f1, "iou": self.iou}


# Sanity check on the convexity relation used in the manuscript discussion.
_m = MicroAccumulator(); _m.tp, _m.fp, _m.fn = 100, 20, 30
assert abs(_m.iou - _m.f1 / (2 - _m.f1)) < 1e-9
print("Metric identity check passed: IoU == F1 / (2 - F1) for micro aggregation.")


Metric identity check passed: IoU == F1 / (2 - F1) for micro aggregation.


## 3. Model definitions

Copied unchanged from the training notebooks so that checkpoints load with strict=True.


In [4]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8, min_mid=False):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        mid = max(ch // r, 4) if min_mid else ch // r
        self.fc = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(True),
                                nn.Linear(mid, ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlockAF(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(ic, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r)
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False), nn.BatchNorm3d(oc))
                     if ic != oc else nn.Identity())

    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.b2(self.c2(o))
        o = self.se(o)
        return self.relu(o + r)


class SEUNet3D(nn.Module):
    """AF variant."""

    def __init__(self, ic=8, nc=1, ec=(64, 128, 256, 512), r=8, dr=0.1):
        super().__init__()
        self.e1 = ResBlockAF(ic, ec[0], r, dr)
        self.e2 = ResBlockAF(ec[0], ec[1], r, dr)
        self.e3 = ResBlockAF(ec[1], ec[2], r, dr)
        self.e4 = ResBlockAF(ec[2], ec[3], r, dr)
        self.pool = nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        self.bot = ResBlockAF(ec[3], ec[3] * 2, r, dr)
        self.u4 = nn.ConvTranspose3d(ec[3] * 2, ec[3], (1, 2, 2), stride=(1, 2, 2))
        self.d4 = ResBlockAF(ec[3] * 2, ec[3], r, dr)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1, 2, 2), stride=(1, 2, 2))
        self.d3 = ResBlockAF(ec[2] * 2, ec[2], r, dr)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1, 2, 2), stride=(1, 2, 2))
        self.d2 = ResBlockAF(ec[1] * 2, ec[1], r, dr)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1, 2, 2), stride=(1, 2, 2))
        self.d1 = ResBlockAF(ec[0] * 2, ec[0], r, dr)
        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.final(d1)


class ResBlockFP(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.conv1 = nn.Conv3d(ic, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.bn1 = nn.BatchNorm3d(oc)
        self.conv2 = nn.Conv3d(oc, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.bn2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, 8, min_mid=True)
        self.skip = nn.Conv3d(ic, oc, 1, bias=False) if ic != oc else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x):
        r = self.skip(x)
        o = self.act(self.bn1(self.conv1(x)))
        o = self.bn2(self.conv2(o))
        return self.act(self.se(o) + r)


class ASPP3D(nn.Module):
    def __init__(self, ic, oc, dils=(1, 6, 12)):
        super().__init__()
        bc = oc // len(dils)
        self.br = nn.ModuleList([
            nn.Sequential(nn.Conv3d(ic, bc, (1, 3, 3), padding=(0, d, d),
                                    dilation=(1, d, d), bias=False),
                          nn.BatchNorm3d(bc), nn.GELU())
            for d in dils])
        self.gp = nn.Sequential(nn.AdaptiveAvgPool3d((None, 1, 1)),
                                nn.Conv3d(ic, bc, 1, bias=False),
                                nn.BatchNorm3d(bc), nn.GELU())
        self.fuse = nn.Sequential(nn.Conv3d(bc * (len(dils) + 1), oc, 1, bias=False),
                                  nn.BatchNorm3d(oc), nn.GELU())

    def forward(self, x):
        parts = [b(x) for b in self.br]
        g = self.gp(x).expand(-1, -1, x.shape[2], x.shape[3], x.shape[4])
        parts.append(g)
        return self.fuse(torch.cat(parts, 1))


class SEUNet3DPred(nn.Module):
    """FP variant."""

    def __init__(self, in_ch=27, enc=(64, 128, 256, 512), bneck=1024):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv3d(in_ch, enc[0], (1, 3, 3), padding=(0, 1, 1), bias=False),
            nn.BatchNorm3d(enc[0]), nn.GELU())
        self.encs = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = enc[0]
        for c in enc:
            self.encs.append(ResBlockFP(prev, c))
            self.pools.append(nn.MaxPool3d((1, 2, 2)))
            prev = c
        self.bneck = ASPP3D(enc[-1], bneck)
        self.ups = nn.ModuleList(); self.decs = nn.ModuleList()
        prev = bneck
        for c in reversed(enc):
            self.ups.append(nn.ConvTranspose3d(prev, c, (1, 2, 2), stride=(1, 2, 2)))
            self.decs.append(ResBlockFP(c * 2, c))
            prev = c
        self.head = nn.Sequential(
            nn.Conv2d(enc[0], 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 1, 1))

    def forward(self, x):
        x = self.inp(x)
        skips = []
        for enc, pool in zip(self.encs, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bneck(x)
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            x = up(x)
            d = [sk.shape[i] - x.shape[i] for i in range(2, 5)]
            if any(di != 0 for di in d):
                x = F.pad(x, [0, d[2], 0, d[1], 0, d[0]])
            x = dec(torch.cat([x, sk], 1))
        x = x.mean(dim=2)
        return self.head(x)


def load_state(model, ckpt_path, device):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ck.get("model_state_dict", ck.get("state_dict", ck))
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    return ck, list(missing), list(unexpected)


## 4. Data access

Two AF label rules are supported so that the effect of the labelling convention
can be separated from the effect of the aggregation:
  "ref"  -> nan_to_num(band7) > 0            (TS-SatFire reference)
  "ours" -> band7 >= 7, NaN frames unlabelled (manuscript)


In [5]:
def read_tif(path):
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)
    arr = tifffile.imread(path).astype(np.float32)
    return arr[np.newaxis] if arr.ndim == 2 else arr


def center_crop(a, size):
    h, w = a.shape[-2], a.shape[-1]
    r0, c0 = (h - size) // 2, (w - size) // 2
    r0, c0 = max(r0, 0), max(c0, 0)
    return a[..., r0:r0 + size, c0:c0 + size]


def day_files(fire_dir):
    return sorted(glob.glob(os.path.join(fire_dir, "VIIRS_Day", "*.tif")))


def load_sat8(fire_dir, dpath):
    d = read_tif(dpath)
    bands = d[:6]
    H, W = bands.shape[1], bands.shape[2]
    npath = os.path.join(fire_dir, "VIIRS_Night",
                         os.path.basename(dpath).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(npath):
        n = read_tif(npath)
        nb = n[:2, :H, :W] if n.shape[0] >= 2 else np.zeros((2, H, W), np.float32)
    else:
        nb = np.zeros((2, H, W), np.float32)
    return np.concatenate([bands, nb], axis=0), d


def af_label_from_raw(raw, rule):
    """raw: full day array. Returns (label, is_unlabelled_frame)."""
    if raw.shape[0] < 7:
        return None, True
    b7 = raw[6]
    all_nan = bool(np.isnan(b7).all())
    if rule == "ref":
        return (np.nan_to_num(b7) > 0).astype(np.float32), all_nan
    return ((np.nan_to_num(b7, nan=0.0) >= 7).astype(np.float32), all_nan)


def ba_mask_from_raw(raw):
    """Training convention: burned = finite band-8 value, NaN = unburned."""
    if raw.shape[0] < 8:
        return None
    return np.isfinite(raw[7]).astype(np.float32)


def ba_day_is_labelled(raw):
    """A day is unlabelled only if band 8 has no finite pixel anywhere."""
    if raw.shape[0] < 8:
        return False
    return bool(np.isfinite(raw[7]).any())


def normalise_sat(stack):
    m = CFG.SAT_MEAN[None, :, None, None]
    s = CFG.SAT_STD[None, :, None, None]
    out = (stack - m) / (s + 1e-8)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


## 5. Per-fire evaluation workers


In [6]:
@torch.no_grad()
def eval_af_fire(model, device, fire_dir, ts, stride, label_rule,
                 ref_thr, our_thr, score_all_frames):
    files = day_files(fire_dir)
    if len(files) < ts:
        return None

    ref_f1, ref_iou = [], []
    n_empty = n_free = 0
    micro_ref_thr = MicroAccumulator()
    micro_our_thr = MicroAccumulator()
    per_frame_our = []

    for t0 in range(0, len(files) - ts + 1, stride):
        frames, raws = [], []
        H = W = None
        for t in range(t0, t0 + ts):
            sat, raw = load_sat8(fire_dir, files[t])
            if H is None:
                H, W = sat.shape[1], sat.shape[2]
            frames.append(sat[:, :H, :W]); raws.append(raw[:, :H, :W])

        x = normalise_sat(np.stack(frames, axis=0))
        x = center_crop(x, CFG.CROP)
        x = torch.from_numpy(x.transpose(1, 0, 2, 3)).float().unsqueeze(0).to(device)

        with torch.autocast("cuda", enabled=True):
            logits = model(x)
        probs = torch.sigmoid(logits[:, 0]).float().cpu().numpy()[0]  # (T,H,W)

        idxs = range(ts) if score_all_frames else [ts - 1]
        for ti in idxs:
            lbl, all_nan = af_label_from_raw(raws[ti], label_rule)
            if lbl is None:
                continue
            lbl = center_crop(lbl, CFG.CROP)
            if label_rule == "ours" and all_nan:
                continue  # manuscript protocol skips unlabelled frames entirely

            p_ref = probs[ti] > ref_thr
            p_our = probs[ti] > our_thr

            f1, iou, empty, free = ref_frame_score(lbl, p_ref)
            ref_f1.append(f1); ref_iou.append(iou)
            n_empty += int(empty); n_free += int(free)

            micro_ref_thr.add(lbl > 0, p_ref)
            micro_our_thr.add(lbl > 0, p_our)
            per_frame_our.append(float(np.sum((p_our) & (lbl > 0))))

    if not ref_f1:
        return None

    return {
        "n_frames": len(ref_f1),
        "n_empty_gt": n_empty,
        "n_free_score": n_free,
        "ref_f1": float(np.mean(ref_f1)),
        "ref_iou": float(np.mean(ref_iou)),
        "micro_at_ref_thr": micro_ref_thr.as_dict(),
        "micro_at_our_thr": micro_our_thr.as_dict(),
    }


@torch.no_grad()
def eval_fp_fire(model, device, fire_dir, ts, ref_thr, our_thr, fp_mean, fp_std):
    """Progression: label = BA(T+1) \\ BA(T); reference stride is 1."""
    files = day_files(fire_dir)
    if len(files) < ts + 1:
        return None

    ref_f1, ref_iou = [], []
    n_empty = n_free = 0
    micro_ref_thr = MicroAccumulator()
    micro_our_thr = MicroAccumulator()

    for t0 in range(0, len(files) - ts):
        frames, raws = [], []
        H = W = None
        ok = True
        for t in range(t0, t0 + ts):
            sat, raw = load_sat8(fire_dir, files[t])
            if H is None:
                H, W = sat.shape[1], sat.shape[2]
            frames.append(sat[:, :H, :W]); raws.append(raw[:, :H, :W])

        nxt = read_tif(files[t0 + ts])
        ba_t = ba_mask_from_raw(raws[-1])
        ba_n = ba_mask_from_raw(nxt[:, :H, :W])
        if ba_t is None or ba_n is None:
            continue

        # Skip the window if either day carries no burned-area label at all.
        if not ba_day_is_labelled(raws[-1]) or not ba_day_is_labelled(nxt[:, :H, :W]):
            continue

        ba_t_c = center_crop(ba_t, CFG.CROP)
        ba_n_c = center_crop(ba_n, CFG.CROP)
        label = np.clip(ba_n_c - ba_t_c, 0, 1).astype(np.float32)

        # Auxiliary channels
        aux = build_fp_aux(fire_dir, files[t0:t0 + ts], raws, H, W, fp_mean, fp_std)
        if aux is None:
            continue
        x = torch.from_numpy(aux).float().unsqueeze(0).to(device)

        with torch.autocast("cuda", enabled=True):
            logits = model(x)
        probs = torch.sigmoid(logits).float().cpu().numpy()[0, 0]

        p_ref = probs > ref_thr
        p_our = probs > our_thr

        f1, iou, empty, free = ref_frame_score(label, p_ref)
        ref_f1.append(f1); ref_iou.append(iou)
        n_empty += int(empty); n_free += int(free)

        valid = label != -1
        micro_ref_thr.add(label > 0, p_ref, valid)
        micro_our_thr.add(label > 0, p_our, valid)

    if not ref_f1:
        return None

    return {
        "n_frames": len(ref_f1),
        "n_empty_gt": n_empty,
        "n_free_score": n_free,
        "ref_f1": float(np.mean(ref_f1)),
        "ref_iou": float(np.mean(ref_iou)),
        "micro_at_ref_thr": micro_ref_thr.as_dict(),
        "micro_at_our_thr": micro_our_thr.as_dict(),
    }


def normalize_firepred(fp, mean, std):
    """Exact copy of the training-time normalisation."""
    m = mean.reshape(-1, 1, 1)
    s = np.maximum(std, 0.1).reshape(-1, 1, 1)    # floor std to avoid blow-up
    out = (fp - m) / s
    out = np.clip(out, -5.0, 5.0)                 # clip extreme z-scores
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def build_fp_aux(fire_dir, dpaths, raws, H, W, fp_mean, fp_std):
    """27 channels: 8 VIIRS + 18 FirePred + 1 per-day BA, matching training."""
    cs = CFG.CROP
    n_fp = len(CFG.FP_BAND_IDX)
    frames = []

    for t_idx, dp in enumerate(dpaths):
        s, _ = load_sat8(fire_dir, dp)
        if s.shape[1] < cs or s.shape[2] < cs:
            return None
        sat_crop = center_crop(normalise_sat(s[None]), cs)[0]      # (8, cs, cs)

        fpath = os.path.join(fire_dir, "FirePred",
                             os.path.basename(dp).replace("_VIIRS_Day", "_FirePred"))
        fp_crop = np.zeros((n_fp, cs, cs), dtype=np.float32)
        if os.path.exists(fpath):
            a = read_tif(fpath)
            if a.shape[0] >= 19:
                a = np.nan_to_num(a[CFG.FP_BAND_IDX], nan=0.0, posinf=0.0, neginf=0.0)
                if fp_mean is not None:
                    a = normalize_firepred(a, fp_mean, fp_std)
                if a.shape[1] >= cs and a.shape[2] >= cs:
                    fp_crop = center_crop(a, cs)

        b = ba_mask_from_raw(raws[t_idx])
        ba_crop = (center_crop(b, cs)[np.newaxis] if b is not None
                   else np.zeros((1, cs, cs), dtype=np.float32))

        frames.append(np.concatenate([sat_crop, fp_crop, ba_crop], axis=0))

    return np.stack(frames, axis=1)      # (27, T, cs, cs)


## 6. Multi-GPU dispatch

Fires are sharded across the available T4s; each worker writes one JSON shard.


In [7]:
def worker(rank, task, fire_ids, cfg_blob, out_dir):
    device = torch.device("cuda", rank)
    torch.cuda.set_device(device)

    if task == "af":
        model = SEUNet3D(ic=CFG.AF_CHANNELS).to(device)
        _, miss, unexp = load_state(model, cfg_blob["ckpt"], device)
    else:
        model = SEUNet3DPred(CFG.FP_CHANNELS).to(device)
        _, miss, unexp = load_state(model, cfg_blob["ckpt"], device)
    model.eval()
    if rank == 0 and (miss or unexp):
        print("[warn] missing keys:", miss[:5], "unexpected:", unexp[:5])

    fp_mean = fp_std = None
    if cfg_blob.get("fp_stats") and os.path.exists(cfg_blob["fp_stats"]):
        z = np.load(cfg_blob["fp_stats"])
        fp_mean, fp_std = z["mean"].astype(np.float32), z["std"].astype(np.float32)

    results = {}
    mine = fire_ids[rank::max(N_GPU, 1)]
    for fid in mine:
        fdir = os.path.join(DATA_ROOT, fid)
        if not os.path.isdir(fdir):
            results[fid] = {"error": "missing directory"}
            continue
        try:
            if task == "af":
                r = eval_af_fire(model, device, fdir,
                                 CFG.AF_TS, cfg_blob["stride"], cfg_blob["label_rule"],
                                 CFG.REF_THRESHOLD, CFG.OUR_AF_THRESHOLD,
                                 cfg_blob["score_all_frames"])
            else:
                r = eval_fp_fire(model, device, fdir, CFG.FP_TS,
                                 CFG.REF_THRESHOLD, CFG.OUR_FP_THRESHOLD,
                                 fp_mean, fp_std)
            results[fid] = r if r is not None else {"error": "no usable window"}
        except Exception as e:
            results[fid] = {"error": "{}: {}".format(type(e).__name__, e)}
        print("[gpu {}] {} done".format(rank, fid), flush=True)

    with open(os.path.join(out_dir, "shard_{}_{}.json".format(task, rank)), "w") as f:
        json.dump(results, f)


def run_task(task, fire_ids, cfg_blob, tag):
    out_dir = os.path.join(OUT_DIR, "shards_" + tag)
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(os.path.join(out_dir, "*.json")):
        os.remove(f)

    t0 = time.time()
    n = max(N_GPU, 1)
    if n > 1:
        with ThreadPoolExecutor(max_workers=n) as ex:
            futures = [ex.submit(worker, r, task, fire_ids, cfg_blob, out_dir)
                       for r in range(n)]
            for f in futures:
                f.result()
    else:
        worker(0, task, fire_ids, cfg_blob, out_dir)

    merged = {}
    for f in sorted(glob.glob(os.path.join(out_dir, "*.json"))):
        merged.update(json.load(open(f)))
    print("{} finished in {:.1f} min".format(tag, (time.time() - t0) / 60))
    return merged


## 7. Aggregation


In [8]:
def aggregate(results, fire_subset):
    """Return reference macro scores and micro scores over a chosen fire subset."""
    used = [f for f in fire_subset if f in results and "error" not in results[f]]
    missing = [f for f in fire_subset if f not in used]

    ref_f1 = float(np.mean([results[f]["ref_f1"] for f in used])) if used else float("nan")
    ref_iou = float(np.mean([results[f]["ref_iou"] for f in used])) if used else float("nan")

    out = {"n_fires": len(used), "missing": missing,
           "ref_macro_f1": ref_f1, "ref_macro_iou": ref_iou,
           "n_frames": sum(results[f]["n_frames"] for f in used),
           "n_empty_gt": sum(results[f]["n_empty_gt"] for f in used),
           "n_free_score": sum(results[f]["n_free_score"] for f in used)}

    for key, name in [("micro_at_ref_thr", "micro_thr050"),
                      ("micro_at_our_thr", "micro_ourthr")]:
        acc = MicroAccumulator()
        for f in used:
            d = results[f][key]
            acc.tp += d["tp"]; acc.fp += d["fp"]; acc.fn += d["fn"]
        out[name + "_f1"] = acc.f1
        out[name + "_iou"] = acc.iou
        out[name + "_perfire_macro_f1"] = float(
            np.mean([results[f][key]["f1"] for f in used])) if used else float("nan")
    return out


def report(title, results, full_list, audited_list, published_f1, published_iou):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    rows = [("full official", full_list), ("audited subset", audited_list)]
    print("{:<16} {:>6} {:>9} {:>9} {:>9} {:>9} {:>9}".format(
        "population", "fires", "refF1", "refIoU", "microF1", "microIoU", "freeFrm"))
    print("-" * 78)
    agg = {}
    for name, lst in rows:
        a = aggregate(results, lst)
        agg[name] = a
        print("{:<16} {:>6d} {:>9.4f} {:>9.4f} {:>9.4f} {:>9.4f} {:>6d}/{:d}".format(
            name, a["n_fires"], a["ref_macro_f1"], a["ref_macro_iou"],
            a["micro_ourthr_f1"], a["micro_ourthr_iou"],
            a["n_free_score"], a["n_frames"]))
    print("-" * 78)
    print("published baseline (reference protocol): F1 = {:.3f}  IoU = {:.3f}".format(
        published_f1, published_iou))
    d = agg["full official"]["ref_macro_f1"] - published_f1
    print("delta on full official set, matched protocol: {:+.4f}".format(d))
    print("\nConsistency note: for micro aggregation IoU must equal F1/(2-F1).")
    m = agg["full official"]
    print("  micro   F1 {:.4f} -> implied IoU {:.4f} (reported {:.4f})".format(
        m["micro_ourthr_f1"], m["micro_ourthr_f1"] / (2 - m["micro_ourthr_f1"]),
        m["micro_ourthr_iou"]))
    print("  ref macro F1 {:.4f} -> implied IoU {:.4f} (reported {:.4f})".format(
        m["ref_macro_f1"], m["ref_macro_f1"] / (2 - m["ref_macro_f1"]),
        m["ref_macro_iou"]))
    return agg


## 8. Run

AF is evaluated in three configurations to separate each protocol difference.
FP is evaluated once over all 24 official test fires; the subsets are formed
afterwards from the per-fire results, so no re-run is needed to change policy.


In [9]:
def per_fire_frame(results, tag):
    rows = []
    for fid, r in results.items():
        if not isinstance(r, dict) or "error" in r:
            rows.append({"config": tag, "fire": fid, "error":
                         r.get("error") if isinstance(r, dict) else "none"})
            continue
        rows.append({
            "config": tag, "fire": fid, "n_frames": r["n_frames"],
            "n_empty_gt": r["n_empty_gt"], "n_free_score": r["n_free_score"],
            "ref_f1": r["ref_f1"], "ref_iou": r["ref_iou"],
            "micro_f1_thr050": r["micro_at_ref_thr"]["f1"],
            "micro_iou_thr050": r["micro_at_ref_thr"]["iou"],
            "micro_f1_ourthr": r["micro_at_our_thr"]["f1"],
            "micro_iou_ourthr": r["micro_at_our_thr"]["iou"],
            "tp": r["micro_at_our_thr"]["tp"], "fp": r["micro_at_our_thr"]["fp"],
            "fn": r["micro_at_our_thr"]["fn"], "error": None})
    return pd.DataFrame(rows)


if __name__ == "__main__":
    import pandas as pd

    af_all = CFG.AF_TEST_ALL
    af_aud = [f for f in af_all if f not in CFG.AF_TEST_EXCLUDE]
    af_aud_nodc = [f for f in af_aud if f != "double_creek_fire"]

    af_configs = {
        "af_reference": dict(ckpt=AF_CKPT, stride=CFG.AF_TS, label_rule="ref",
                             score_all_frames=True),
        "af_ourlabel": dict(ckpt=AF_CKPT, stride=CFG.AF_TS, label_rule="ours",
                            score_all_frames=True),
        "af_lastframe": dict(ckpt=AF_CKPT, stride=1, label_rule="ours",
                             score_all_frames=False),
    }

    af_out, af_frames = {}, []
    for tag, blob in af_configs.items():
        res = run_task("af", af_all, blob, tag)
        json.dump(res, open(os.path.join(OUT_DIR, tag + "_perfire.json"), "w"),
                  indent=2, default=str)
        af_frames.append(per_fire_frame(res, tag))
        af_out[tag] = report(
            "ACTIVE FIRE -- {} (stride={}, label={}, all_frames={})".format(
                tag, blob["stride"], blob["label_rule"], blob["score_all_frames"]),
            res, af_all, af_aud, published_f1=0.823, published_iou=0.727)
        if tag == "af_lastframe":
            print("\nSensitivity to double_creek_fire (3/10 days labelled):")
            a_with = aggregate(res, af_aud)
            a_without = aggregate(res, af_aud_nodc)
            print("  15 fires  micro F1 {:.4f}   ref macro F1 {:.4f}".format(
                a_with["micro_ourthr_f1"], a_with["ref_macro_f1"]))
            print("  14 fires  micro F1 {:.4f}   ref macro F1 {:.4f}".format(
                a_without["micro_ourthr_f1"], a_without["ref_macro_f1"]))
            af_out["double_creek_sensitivity"] = {"with": a_with, "without": a_without}

    pd.concat(af_frames, ignore_index=True).to_csv(
        os.path.join(OUT_DIR, "af_per_fire_all_configs.csv"), index=False)

    # ---- Fire progression, all 24 official test fires ----
    fp_all = sorted([d for d in os.listdir(DATA_ROOT) if d.startswith("US_2021")])
    fp_16 = [f for f in fp_all if f not in CFG.FP_TEST_EXCLUDE]
    RULE_A_EXCLUDE = [
        "US_2021_FL2521008104520210308", "US_2021_NM3340210587120210426",
        "US_2021_NM3676810505920211120", "US_2021_NM3344410803520210514",
        "US_2021_NM3323810847220210520", "US_2021_MT4714310953420211004",
    ]
    fp_18 = [f for f in fp_all if f not in RULE_A_EXCLUDE]
    print("\nFP test fires:", len(fp_all), "| manuscript subset:", len(fp_16),
          "| Rule-A subset:", len(fp_18))

    fp_stats = None
    for pat in ["/kaggle/input/**/fp_stats*.npz", "/kaggle/working/**/fp_stats*.npz"]:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            fp_stats = hits[0]; break
    print("FP normalisation stats:", fp_stats)
    assert fp_stats, "fp_stats*.npz not found; FP channels would be unnormalised"

    res_fp = run_task("fp", fp_all, dict(ckpt=FP_CKPT, fp_stats=fp_stats),
                      "fp_reference")
    json.dump(res_fp, open(os.path.join(OUT_DIR, "fp_reference_perfire.json"), "w"),
              indent=2, default=str)
    per_fire_frame(res_fp, "fp_reference").to_csv(
        os.path.join(OUT_DIR, "fp_per_fire.csv"), index=False)

    fp_agg = report("FIRE PROGRESSION -- all 24 official test fires vs manuscript 16",
                    res_fp, fp_all, fp_16, published_f1=0.375, published_iou=0.338)

    print("\nFP aggregates under each exclusion policy:")
    for name, lst in [("all 24 (recommended)", fp_all),
                      ("Rule A, 18 fires", fp_18),
                      ("manuscript, 16 fires", fp_16)]:
        a = aggregate(res_fp, lst)
        print("  {:<24} n={:2d}  ref macro F1 {:.4f}   micro F1 {:.4f}".format(
            name, a["n_fires"], a["ref_macro_f1"], a["micro_ourthr_f1"]))
        fp_agg[name] = a

    json.dump({"af": af_out, "fp": fp_agg},
              open(os.path.join(OUT_DIR, "protocol_summary.json"), "w"),
              indent=2, default=str)

    print("\nFiles written:")
    for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv")) +
                    glob.glob(os.path.join(OUT_DIR, "*.json"))):
        print("  {:>8.1f} KB  {}".format(os.path.getsize(f) / 1e3, f))
    print("\nSend me the printed AF and FP tables, plus the two lines above them.")


[gpu 0] elephant_hill_fire done
[gpu 1] eagle_bluff_fire done
[gpu 0] double_creek_fire done
[gpu 1] sparks_lake_fire done
[gpu 0] lytton_fire done
[gpu 1] chuckegg_creek_fire done
[gpu 0] swedish_fire done
[gpu 1] sydney_fire done
[gpu 0] thomas_fire done
[gpu 1] tubbs_fire done
[gpu 0] carr_fire done
[gpu 1] camp_fire done
[gpu 0] creek_fire done
[gpu 1] blue_ridge_fire done
[gpu 0] dixie_fire done
[gpu 1] mosquito_fire done
[gpu 0] calfcanyon_fire done
af_reference finished in 0.4 min

ACTIVE FIRE -- af_reference (stride=2, label=ref, all_frames=True)
population        fires     refF1    refIoU   microF1  microIoU   freeFrm
------------------------------------------------------------------------------
full official        17    0.6912    0.6044    0.8180    0.6920     20/170
audited subset       15    0.7567    0.6583    0.8298    0.7091     16/150
------------------------------------------------------------------------------
published baseline (reference protocol): F1 = 0.823  IoU 